# CrewAI — Intermediate Agent

**Framework:** CrewAI  
**Level:** 02 — Intermediate  
**Model:** `gemini-2.0-flash`

### What's New vs Simple
| Feature | Simple | Intermediate |
|---|---|---|
| Agents | 1 | **2** (Researcher + Formatter) |
| Tasks | 1 | **2 with dependency** (`context=[prev]`) |
| Tools | On single agent | **Only on Researcher** (role separation) |
| Output | Free text | **Structured** (`output_pydantic=TravelBriefing`) |

## Concept: Multi-Agent Task Pipeline

```
┌─────────────────────────────────────────────────────────┐
│                        CREW                             │
│                                                         │
│  ┌───────────────────────┐   ┌──────────────────────┐   │
│  │  Researcher           │   │  Formatter           │   │
│  │  tools: weather,      │   │  tools: none         │   │
│  │         time,         │   │  works only with     │   │
│  │         advisory      │   │  research output     │   │
│  └──────────┬────────────┘   └──────────┬───────────┘   │
│             │                           │               │
│  ┌──────────▼────────────┐              │               │
│  │  Task 1: Research     │              │               │
│  │  "Gather data for X" │──context──▶  │               │
│  └───────────────────────┘  ┌──────────▼───────────┐   │
│                             │  Task 2: Format      │   │
│                             │  output_pydantic=    │   │
│                             │  TravelBriefing      │   │
│                             └──────────────────────┘   │
│                                                         │
│  Process: Sequential                                    │
└─────────────────────────────────────────────────────────┘
```

**Key CrewAI concepts at this level:**
- **Task `context`** — passes one task's output as input to another
- **Role separation** — Researcher uses tools; Formatter only processes text
- **`output_pydantic`** — the final task returns a validated Pydantic instance

**Compare to LangChain:** LangChain uses `RunnableWithMessageHistory` for conversational memory. CrewAI's 'memory' is task context — structured output from one task feeding the next.

## Setup

In [ ]:
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env"
print("✓ Ready")

## Structured Output Schema

In [ ]:
class TravelBriefing(BaseModel):
    city: str = Field(description="Name of the city")
    weather_summary: str = Field(description="Weather conditions and temperature")
    local_time: str = Field(description="Current local time with timezone")
    travel_advisory: str = Field(description="Safety advisory level and key notes")
    recommendation: str = Field(description="One-sentence travel recommendation")

print("Schema:", list(TravelBriefing.model_fields.keys()))

## 3 Tools — Assigned Only to the Researcher

In [ ]:
@tool("Weather Tool")
def get_weather(city: str) -> str:
    """Get current weather for a city. Input: city name."""
    data = {
        "london":    ("Cloudy",       12, 78), "new york":  ("Sunny",         22, 45),
        "bangalore": ("Rainy",        25, 85), "tokyo":     ("Clear",         18, 60),
        "paris":     ("Partly Cloudy",16, 65),
    }
    key = city.lower()
    if key in data:
        cond, temp, hum = data[key]
        return f"{city}: {cond}, {temp}°C, humidity {hum}%."
    return f"No data for '{city}'."


@tool("Time Tool")
def get_time(city: str) -> str:
    """Get current local time for a city. Input: city name."""
    times = {
        "london": "14:30 GMT", "new york": "09:30 EST",
        "bangalore": "20:00 IST", "tokyo": "22:30 JST", "paris": "15:30 CET",
    }
    key = city.lower()
    return f"Current time in {city}: {times.get(key, 'Unknown')}"


@tool("Travel Advisory Tool")
def get_travel_advisory(city: str) -> str:  # ← NEW
    """Get travel safety advisory for a city. Input: city name."""
    advisories = {
        "london":    ("Low",    "Routine precautions."),
        "new york":  ("Low",    "Normal precautions."),
        "bangalore": ("Medium", "Monsoon affects transport."),
        "tokyo":     ("Low",    "Very safe city."),
        "paris":     ("Low",    "Alert in crowded spots."),
    }
    key = city.lower()
    if key in advisories:
        level, notes = advisories[key]
        return f"{city} Advisory: Level {level}. {notes}"
    return f"No advisory data for '{city}'."


print("3 tools defined")

## Two Agents

**Key design decision:** The Formatter has `tools=[]`. It only writes — never calls APIs. This separation of concerns mirrors how you'd structure a real team: data collectors vs. communicators.

In [ ]:
gemini = LLM(model="gemini/gemini-2.0-flash", temperature=0)

researcher = Agent(
    role="Travel Intelligence Researcher",
    goal="Gather comprehensive, accurate travel data for a given city using all available tools.",
    backstory=(
        "You are a meticulous travel data analyst who always checks weather, local time, "
        "and safety advisories before compiling a city report. You never skip a data source."
    ),
    tools=[get_weather, get_time, get_travel_advisory],  # ← has tools
    llm=gemini,
    verbose=False,
)

formatter = Agent(
    role="Travel Briefing Specialist",
    goal="Transform raw travel research data into a polished, actionable travel briefing.",
    backstory=(
        "You are a professional travel writer who takes raw data from research analysts "
        "and crafts clear, concise briefings that help travelers make informed decisions."
    ),
    tools=[],  # ← no tools — relies entirely on researcher's output
    llm=gemini,
    verbose=False,
)

print(f"Researcher: {researcher.role} | Formatter: {formatter.role}")

## Tasks with Dependency

`context=[research_task]` tells CrewAI: before running `format_task`, inject the output of `research_task` as additional context. This is how CrewAI chains information.

In [ ]:
def build_crew(city: str) -> Crew:
    research_task = Task(
        description=f"Research the city '{city}'. Collect: current weather, local time, and travel advisory.",
        expected_output=f"Raw data for {city}: weather conditions + temperature, local time, safety advisory.",
        agent=researcher,
    )

    format_task = Task(
        description=(
            f"Using the research data provided, create a structured travel briefing for {city}. "
            "Fill all fields of the TravelBriefing schema accurately."
        ),
        expected_output="A complete TravelBriefing with all fields populated.",
        agent=formatter,
        context=[research_task],       # ← passes researcher's output to formatter
        output_pydantic=TravelBriefing, # ← final output is a TravelBriefing instance
    )

    return Crew(
        agents=[researcher, formatter],
        tasks=[research_task, format_task],
        process=Process.sequential,
        verbose=False,
    )

print("Crew factory ready")

## Run

In [ ]:
# Run for Tokyo
crew = build_crew("Tokyo")
result = crew.kickoff()

briefing = result.pydantic
if briefing:
    print(f"City:           {briefing.city}")
    print(f"Weather:        {briefing.weather_summary}")
    print(f"Time:           {briefing.local_time}")
    print(f"Advisory:       {briefing.travel_advisory}")
    print(f"Recommendation: {briefing.recommendation}")
else:
    print("Raw output:", result.raw)

In [ ]:
# Run for Bangalore
crew2 = build_crew("Bangalore")
result2 = crew2.kickoff()

briefing2 = result2.pydantic
if briefing2:
    print(f"City:           {briefing2.city}")
    print(f"Advisory:       {briefing2.travel_advisory}")
    print(f"Recommendation: {briefing2.recommendation}")
else:
    print("Raw:", result2.raw)

In [ ]:
# Side-by-side comparison
if briefing and briefing2:
    print(f"{'Field':<20} {'Tokyo':<35} {'Bangalore'}")
    print("-" * 80)
    for field in ["weather_summary", "local_time", "travel_advisory"]:
        v1 = getattr(briefing, field, 'N/A')[:30]
        v2 = getattr(briefing2, field, 'N/A')[:30]
        print(f"{field:<20} {v1:<35} {v2}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Two agents with different roles | Role separation = cleaner responsibilities, easier to test each agent |
| `context=[research_task]` | Explicit data flow — no magic, you see exactly what gets passed |
| Formatter has `tools=[]` | Not all agents need tools — some only transform/write |
| `output_pydantic=TravelBriefing` | Final output is a validated Python object, not raw text |
| `result.pydantic` | Access the structured output from `CrewOutput` |

### CrewAI 'Memory' vs Other Frameworks
CrewAI's memory in the Intermediate level is **task context** — passing outputs forward, not conversation history. For persistent memory across crew runs, CrewAI supports a `Memory` module (`long_term_memory`, `short_term_memory`) — covered in the Complex level.

### Next: [03-Complex →](../03-complex/agent.ipynb)